In [1]:
%pip install deap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.0/635.0 kB 15.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [deap]
Note: you may need to restart the kernel to use updated packages.


In [1]:
def read_file(path):
    value_list = []
    weight_list = []
    with open(path) as f:    
        num_items, capacity = map(int, f.readline().split())
        for line in range(int(num_items)):
            (value, weight) = map(int,f.readline().split())
            value_list.append(value)
            weight_list.append(weight)
    return (num_items, capacity, value_list, weight_list)

In [2]:
(num_items, capacity, value_list, weight_list) = read_file("./dataset/question_1/knapsack-data/10_269")
print("Values are:",value_list)
print("Weights are:",weight_list)

Values are: [55, 10, 47, 5, 4, 50, 8, 61, 85, 87]
Weights are: [95, 4, 60, 32, 23, 72, 80, 62, 65, 46]


## Importing the Required packages

In [3]:
import random
import math
import numpy as np
import pandas as pd
from deap import base, creator, tools

In [66]:
# Creating Fitness and Individual Class
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

# Mantaining all the constants that will be used internally in the code
IND_SIZE = num_items
POPULATION_SIZE = 50
NUM_OF_GENERATION = 100
VIOLATION_FACTOR = 5
MUTATION_RATE = 0.20
ELITE_SIZE = max(2, int(0.05*POPULATION_SIZE))

# Mutation: Flip Bit with independent probability of 20%
def mutate_child(child):
    if random.random() < MUTATION_RATE:
        # Perform flip mutation on the child
        index = random.randrange(len(child))
        child[index] = 1 - child[index]
    return child

# Evaluation Strategy for each individual
def do_evaluation(individual):
    # Calcualting the bag value and weight for a individual
    (total_bag_value, total_bag_weight) = calculate_value_and_weight(individual)
    # Caclulating the vilation value
    violation = max(0,  total_bag_weight - capacity)
    # Final Fitness vlaue calculation
    fitness = total_bag_value - VIOLATION_FACTOR * (violation)
    return (fitness, )

# Function to calculate value and weight for one individual chromosome
def calculate_value_and_weight(individual):
    value, weight = 0, 0
    for index, item in enumerate(individual):
        value += item * value_list[index]
        weight += item * weight_list[index]
    return (value, weight)

# Registering function with the toolbox
toolbox = base.Toolbox()
toolbox.register("attr_binary", random.choice, [0, 1])
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_binary, n=IND_SIZE)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
# Evaluation: Custom Method for evaluation
toolbox.register("evaluate", do_evaluation)
# Selection: K-Tournament approach
toolbox.register("select", tools.selTournament, tournsize=3)
# Crossover: Two Point crossover approach
toolbox.register("crossover", tools.cxTwoPoint)
# Mutation: Custom method with 20% Mutation rate for each individual
toolbox.register('mutate', mutate_child)

In [42]:
# Initializing a initial population
initial_population = toolbox.population(POPULATION_SIZE)

In [67]:
# Now here we need to add a loop
for i in range(NUM_OF_GENERATION):
    new_population = list()

    for individual in initial_population:
        individual.fitness.values = toolbox.evaluate(individual)
    
    # Performing Elitism so that we have the best individual carried forward
    elites = tools.selBest(initial_population, ELITE_SIZE)
    elites = list(map(toolbox.clone, elites))
    new_population.extend(elites)
    
    while(len(new_population)<POPULATION_SIZE):
        # Selection: K-Tournament approach - choose two individual for crossover
        offspring = toolbox.select(individuals = initial_population, k = 2)
        # Clone the selected individuals
        [parent1, parent2] = map(toolbox.clone, offspring)
        
        # Crossover: Two Point crossover approach
        [child1, child2] = toolbox.crossover(parent1, parent2)
        
        # Mutation: Custom method with 20% Mutation rate for each individual
        child1 = toolbox.mutate(child1)
        child2 = toolbox.mutate(child2)

        child1.fitness.values = toolbox.evaluate(child1)
        child2.fitness.values = toolbox.evaluate(child2)
        
        if (len(new_population) < POPULATION_SIZE):
            new_population.append(child1)
            
        if (len(new_population) < POPULATION_SIZE):
            new_population.append(child2)

    initial_population = new_population

In [70]:
for individual in new_population:
    print(toolbox.evaluate(individual))

(190,)
(-365,)
(-320,)
(62,)
(-524,)
(102,)
(30,)
(17,)
(-39,)
(283,)
(-15,)
(156,)
(204,)
(201,)
(181,)
(178,)
(-391,)
(112,)
(98,)
(-223,)
(70,)
(-293,)
(144,)
(-41,)
(14,)
(30,)
(191,)
(-223,)
(-463,)
(30,)
(150,)
(126,)
(154,)
(144,)
(-41,)
(155,)
(273,)
(-182,)
(158,)
(85,)
(-42,)
(-375,)
(90,)
(-463,)
(167,)
(191,)
(-320,)
(166,)
(166,)
(85,)
